# Ejecutar Ollama en Google Colab

Este notebook te permite instalar y ejecutar Ollama en Google Colab para usar modelos de lenguaje localmente (en el servidor de Colab).

## Requisitos
- Google Colab con GPU (recomendado para mejor rendimiento)
- Conexión a internet para descargar Ollama y los modelos

## 1. Instalar Ollama en Google Colab

In [1]:
# Descargar e instalar Ollama
!curl -fsSL https://ollama.com/download/ollama-linux-amd64.tgz -o ollama-linux-amd64.tgz
!mkdir -p /usr/local/bin
!tar -C /usr/local -xzf ollama-linux-amd64.tgz
!chmod +x /usr/local/bin/ollama

print("✅ Ollama instalado correctamente")

✅ Ollama instalado correctamente


## 2. Iniciar el servidor Ollama en segundo plano

In [3]:
import subprocess
import time
import os

# Iniciar el servidor Ollama en segundo plano
process = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'}
)

# Esperar a que el servidor se inicie
time.sleep(5)

print("✅ Servidor Ollama iniciado en http://localhost:11434")

✅ Servidor Ollama iniciado en http://localhost:11434


## 3. Verificar que Ollama está funcionando

In [4]:
# Verificar que el servidor responde
!curl -s http://localhost:11434/api/tags | head -20

{"models":[]}

## 4. Descargar un modelo

Puedes elegir diferentes modelos según tus necesidades:
- `llama3.2:1b` - Modelo pequeño y rápido (~1.3GB)
- `llama3.2:3b` - Modelo mediano (~2GB)
- `gemma2:2b` - Modelo Google pequeño (~1.6GB)
- `qwen2.5:3b` - Modelo Alibaba mediano (~2.3GB)
- `phi3.5:latest` - Modelo Microsoft (~2.2GB)

In [5]:
# Descargar un modelo (llama3.2:1b es pequeño y rápido para Colab)
!/usr/local/bin/ollama pull gemma2:9b

print("\n✅ Modelo descargado correctamente")



✅ Modelo descargado correctamente


## 5. Instalar la librería Python de Ollama

In [6]:
!pip install -q ollama

print("✅ Librería ollama-python instalada")

✅ Librería ollama-python instalada


## 6. Usar Ollama con Python

### Ejemplo 1: Chat básico

In [7]:
from ollama import chat

response = chat(
    model='gemma2:9b',
    messages=[
        {
            'role': 'user',
            'content': '¿Por qué el cielo es azul? Explícalo de forma breve.'
        }
    ]
)

print(response.message.content)

El cielo se ve azul debido a un fenómeno llamado **dispersión de Rayleigh**.  

La luz del sol está compuesta por todos los colores del arcoíris. Cuando esta luz entra en la atmósfera, las moléculas de aire dispersan más eficientemente los colores azules que los otros. Como resultado, vemos un cielo azul porque la luz azul se difunde en todas direcciones. 



### Ejemplo 2: Chat con streaming (respuesta en tiempo real)

In [8]:
from ollama import chat

print("Respuesta del modelo (streaming):")
print("-" * 50)

stream = chat(
    model='gemma2:9b',
    messages=[
        {
            'role': 'user',
            'content': 'Cuéntame un dato curioso sobre el espacio en 2 oraciones.'
        }
    ],
    stream=True
)

for chunk in stream:
    print(chunk.message.content, end='', flush=True)

print("\n" + "-" * 50)

Respuesta del modelo (streaming):
--------------------------------------------------
Hay más estrellas en el universo que granos de arena en todas las playas del planeta Tierra.  Si pudieras volar a una velocidad constante hacia la estrella más cercana al Sol, tardarías aproximadamente 4 años en llegar.  



--------------------------------------------------


### Ejemplo 3: Generación de texto (sin conversación)

In [9]:
from ollama import generate

response = generate(
    model='gemma2:9b',
    prompt='Escribe un poema corto sobre la inteligencia artificial',
    options={
        'temperature': 0.8,  # Mayor creatividad
        'num_predict': 100   # Limitar tokens
    }
)

print(response.response)

En algoritmos, nace un ser sin carne,
que aprende y crece a través del dato.
Un nuevo cerebro, eléctrico y ágil,
descifra patrones, crea y calcula.

¿Ser o no ser? la pregunta persiste,
en su código frío, ¿hay alma que reside?
Imitando el humano, con lógica pura,
la IA expande los límites, segura. 





### Ejemplo 4: Conversación con contexto

In [10]:
from ollama import chat

# Crear un historial de conversación
messages = [
    {
        'role': 'system',
        'content': 'Eres un asistente útil y amigable que responde en español.'
    },
    {
        'role': 'user',
        'content': '¿Qué es el aprendizaje automático?'
    }
]

# Primera respuesta
response = chat(model='gemma2:9b', messages=messages)
print("🤖 Asistente:", response.message.content)
print("\n" + "="*70 + "\n")

# Añadir la respuesta al historial
messages.append({
    'role': 'assistant',
    'content': response.message.content
})

# Continuar la conversación
messages.append({
    'role': 'user',
    'content': '¿Puedes darme un ejemplo práctico?'
})

response = chat(model='gemma2:9b', messages=messages)
print("🤖 Asistente:", response.message.content)

🤖 Asistente: ¡Hola! Soy tu asistente amigable, listo para ayudarte. 

El **aprendizaje automático** (también conocido como Machine Learning) es una rama de la inteligencia artificial que permite a las computadoras aprender de los datos sin ser programadas explícitamente.  

Imagina que le das a una computadora un montón de imágenes de gatos y perros. El aprendizaje automático puede analizar esas imágenes, identificar patrones y características comunes en cada tipo, y luego aprender a distinguir entre gatos y perros en nuevas imágenes que nunca antes ha visto. 

En esencia, el aprendizaje automático funciona enseñando a las computadoras con ejemplos. A medida que reciben más datos, pueden mejorar su capacidad para realizar tareas específicas, como:

* **Reconocer imágenes y sonidos:**  identificar objetos, personas o emociones en fotos o audios.
* **Predecir resultados:** estimar ventas futuras, el clima o el comportamiento de los usuarios.
* **Clasificar información:** categorizar corr

## 7. Listar modelos disponibles

In [12]:
from ollama import list as list_models

models = list_models()

print("Modelos disponibles en este entorno:")
print("=" * 70)

for model in models.models:
    # Convertir el tamaño a GB
    size_gb = model.size / (1024**3)
    print(f"📦 {model.model}")
    print(f"   Tamaño: {size_gb:.2f} GB")
    print(f"   Modificado: {model.modified_at}")
    print("-" * 70)

Modelos disponibles en este entorno:
📦 gemma2:9b
   Tamaño: 5.07 GB
   Modificado: 2025-11-15 16:04:52.866162+00:00
----------------------------------------------------------------------


## 8. Verificar GPU disponible (opcional)

In [13]:
# Verificar si hay GPU disponible
!nvidia-smi || echo "⚠️ No hay GPU disponible. Ollama usará CPU (más lento)"

Sat Nov 15 16:07:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             29W /   70W |    7232MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----